In [2]:
# --- Бібліотеки для даної роботи ---
try:
    import numpy, pandas, matplotlib, plotly, sklearn, jupyterlab, ipywidgets, tqdm, pycountry
    print("Бібліотеки вже встановлені. Пропускаємо інсталяцію.")
except ImportError:
    print("Встановлюємо бібліотеки...")
    %pip install -q numpy pandas matplotlib plotly scikit-learn "jupyterlab>=3" "ipywidgets>=7.6" tqdm pycountry

Бібліотеки вже встановлені. Пропускаємо інсталяцію.


## Домашнє завдання: Тема 10. EM-алгоритм та розділення суміші Гаусівських функцій

### **Це допоможе закріпити такі навички:**

- Попередньої підготовки даних до моделювання
- Роботу з бібліотеками аналізу даних

### **Завдання (крок за кроком):**

***Для цієї задачі необхідно буде завантажити дані [World Happiness Report](https://www.kaggle.com/datasets/unsdsn/world-happiness).***

Для виконання завдання необхідно виконати такі кроки:

1. **Інсталювати та імпортувати необхідні бібліотеки:** 
    - Необхідно буде інсталювати такі пакети:
	```bash
	!pip install plotly==5.20.0
	!pip install "jupyterlab>=3" "ipywidgets>=7.6"
	```

2. **Завантажити дані:**
    - З набору https://www.kaggle.com/datasets/unsdsn/world-happiness.
	```bash
	!wget -O WorldHappinessReport.zip https://github.com/goitacademy/NUMERICAL-PROGRAMMING-IN-PYTHON/blob/main/WorldHappinessReport.zip?raw=true
	```

3. **Розпакувати дані:**
    ```bash
	!unzip WorldHappinessReport.zip
	```

4. **Прочитати дані та відобразити загальну інформацію про:**
	- Статистики
	- Типи ознак

5. **Побудувати діаграми розподілу числових ознак:**
    - Проаналізувати на відповідність чи не відповідність нормальному розподілу.

6. **Відібрати числових ознак та кореляційну матрицю:**
    - Виходячи із розуміння домену та даних відібрати певну кількість числових ознак
    - Відобразити кореляційну матрицю (*див. Тема 4. Вимірювання відстаней та подібностей в аналізі даних*)

7. **Зробити висновок про:**
    - Наявність та силу лінійного зв'язку між ознаками.

8. **Відобразити розподіл:**
    - Цільової ознаки (Happiness.Score або Happiness.Rank) за країнами.
    - Використовуючи наведений нижче код для побудови теплової мапи.
	```py
	fig = px.choropleth(data_dataframe,
						locations = "Country",
						color = "Happiness.Score",
						locationmode = "country names",
                    	)
	fig.update_layout(title = "Happiness Index 2017")
	fig.show()
	```

9. **Застосувати стандартизацію даних:**
    - Для приведення всіх значень до одного діапазону статистик.
    - Використовуючи функцію data_scale() та наступні перетворення
	```py
	def data_scale(data, scaler_type='minmax'):
	    from sklearn.preprocessing import MinMaxScaler
	    from sklearn.preprocessing import StandardScaler
	    from sklearn.preprocessing import Normalizer
	    if scaler_type == 'minmax':
	        scaler = MinMaxScaler()
	    if scaler_type == 'std':
	        scaler = StandardScaler()
	    if scaler_type == 'norm':
	        scaler = Normalizer()

	    scaler.fit(data)
	    res = scaler.transform(data)
	    return res

	data_scaled = data_scale(original_dataframe)
	df_scaled = pd.DataFrame(data_scaled, columns=[original_dataframe.columns])
	print(df_scaled.head())
	```

10. **Відобразити статистики:**
    - Отриманого стандартизованого набору даних та порівняти зі статистиками оригінального набору даних.
    - Зробити висновки.

11. **Побудувати модель кластеризації:**
	- Засобами функції `GaussianMixture()` бібліотеки `sklearn`.

12. **Побудувати теплову мапу:**
    - Для відображення розподілу країн за кластерами.

13. **Дослідити вплив:**
    - Різного набору ознак
    - Результат кластеризації

14. **Висновок:**
    - Зробити загальний висновок про відповідність результатів кластеризації оригінальному розподілу країн за ознакою.

**1. Імпорт необхідних бібліотек:**

In [3]:
# 1. СТАНДАРТНІ БІБЛІОТЕКИ PYTHON (Мережа, Файлова система, Попередження)
import os
import shutil
from sklearn.exceptions import ConvergenceWarning
import urllib.request
import zipfile
import warnings

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# 2. РОБОТА З ДАНИМИ ТА МАТЕМАТИКА
import math
import numpy as np
import pandas as pd

# 3. МАШИННЕ НАВЧАННЯ (Кластеризація та Препроцесинг)
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import MinMaxScaler, StandardScaler, Normalizer

# 4. MLOps ТА СЕРІАЛІЗАЦІЯ МОДЕЛЕЙ
import joblib

# 5. ВІЗУАЛІЗАЦІЯ ТА UI (Plotly, IPywidgets & HTML)
from IPython.display import HTML, display
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pycountry
from scipy.stats import multivariate_normal
import scipy.stats as stats

print("📦 Модулі архітектури імпортовано успішно!")

📦 Модулі архітектури імпортовано успішно!


**1.3. Конфігурація експерименту (Глобальні змінні):**

In [31]:
# 1. МЕРЕЖА ТА ФАЙЛОВА СИСТЕМА (MASTER SWITCH)
TARGET_YEAR = "2023"  # Доступні роки: "2015", "2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024"

# Офіційні набори даних (2015-2019) лежать в одному архіві. Сучасні (2020+) публікуються окремо.
KAGGLE_SOURCES = {
    "2015": "unsdsn/world-happiness",
    "2016": "unsdsn/world-happiness",
    "2017": "unsdsn/world-happiness",
    "2018": "unsdsn/world-happiness",
    "2019": "unsdsn/world-happiness",
    "2020": "mathurinache/world-happiness-report",
    "2021": "ajaypalsinghlo/world-happiness-report-2021",
    "2022": "mathurinache/world-happiness-report-2022",
    "2023": "ajaypalsinghlo/world-happiness-report-2023",
    "2024": "ajaypalsinghlo/world-happiness-report-2024"
}

DATA_DIR     = "WorldHappinessDataSet"       # Папка для ізольованого збереження всіх сирих даних
DATASET_URL  = f"https://www.kaggle.com/api/v1/datasets/download/{KAGGLE_SOURCES[TARGET_YEAR]}"
ZIP_PATH     = os.path.join(DATA_DIR, f"dataset_{TARGET_YEAR}.zip")
CSV_FILENAME = os.path.join(DATA_DIR, f"{TARGET_YEAR}.csv") # Динамічний шлях до потрібного файлу
EXPORT_CSV_PATH = os.path.join(DATA_DIR, f"{TARGET_YEAR}_clustered_result.csv") # Шлях для збереження фінальної таблиці з мітками ШІ

# 2. СТРУКТУРА ДАНИХ ТА ОЗНАКИ (Вирішення проблеми Schema Drift)
SCHEMA_MAPPING = {
    # --- СТАРА ЕРА (2015-2017) ---
    "2015": {
        "country": "Country",
        "target": "Happiness Score",
        "features": ["Economy (GDP per Capita)", "Family", "Health (Life Expectancy)", "Freedom", "Trust (Government Corruption)"]
    },
    "2016": {
        "country": "Country",
        "target": "Happiness Score",
        "features": ["Economy (GDP per Capita)", "Family", "Health (Life Expectancy)", "Freedom", "Trust (Government Corruption)"]
    },
    "2017": {
        "country": "Country",
        "target": "Happiness.Score",
        "features": ["Economy..GDP.per.Capita.", "Family", "Health..Life.Expectancy.", "Freedom", "Trust..Government.Corruption."]
    },

    # --- ПЕРЕХІДНА ЕРА (2018-2019) ---
    "2018": {
        "country": "Country or region",
        "target": "Score",
        "features": ["GDP per capita", "Social support", "Healthy life expectancy", "Freedom to make life choices", "Perceptions of corruption"]
    },
    "2019": {
        "country": "Country or region",
        "target": "Score",
        "features": ["GDP per capita", "Social support", "Healthy life expectancy", "Freedom to make life choices", "Perceptions of corruption"]
    },

    # --- СУЧАСНА ЕРА (2020-2024) ---
    "2020": {
        "country": "Country name",
        "target": "Ladder score",
        "features": ["Logged GDP per capita", "Social support", "Healthy life expectancy", "Freedom to make life choices", "Perceptions of corruption"]
    },
    "2021": {
        "country": "Country name",
        "target": "Ladder score",
        "features": ["Logged GDP per capita", "Social support", "Healthy life expectancy", "Freedom to make life choices", "Perceptions of corruption"]
    },
    "2022": {
        "country": "Country",
        "target": "Happiness score", # У 2022 році Kaggle-автор трохи змінив регістр
        "features": ["Explained by: GDP per capita", "Explained by: Social support", "Explained by: Healthy life expectancy", "Explained by: Freedom to make life choices", "Explained by: Perceptions of corruption"]
    },
    "2023": {
        "country": "Country name",
        "target": "Ladder score",
        "features": ["Logged GDP per capita", "Social support", "Healthy life expectancy", "Freedom to make life choices", "Perceptions of corruption"]
    },
    "2024": {
        "country": "Country name",
        "target": "Ladder score",
        "features": ["Logged GDP per capita", "Social support", "Healthy life expectancy", "Freedom to make life choices", "Perceptions of corruption"]
    }
}

CURRENT_SCHEMA   = SCHEMA_MAPPING[TARGET_YEAR]
COUNTRY_COL      = CURRENT_SCHEMA["country"]             # Динамічна колонка країни (змінювалась у 2018 та 2020)
TARGET_METRIC    = CURRENT_SCHEMA["target"]              # Головна цільова метрика (Індекс щастя)
FEATURES_FULL    = CURRENT_SCHEMA["features"]            # Повний набір соціально-економічних ознак для GMM
FEATURES_MINI    = [FEATURES_FULL[0], FEATURES_FULL[2]]  # Зменшений набір (ВВП та Здоров'я) для дослідження розмірності


# 3. МАШИННЕ НАВЧАННЯ (GMM) ТА СЕРІАЛІЗАЦІЯ
N_CLUSTERS       = 3          # Кількість кластерів: задає число прихованих Гаусівських розподілів (Високий, Середній, Низький рівень)
COVARIANCE_TYPE  = 'full'     # Форма матриці коваріації (геометрія кластерів): 'full' - різні еліпси під будь-яким кутом | 'tied' - однакова форма та нахил для всіх | 'diag' - еліпси строго паралельні осям координат | 'spherical' - ідеальні круглі сфери різного радіусу
GMM_INIT_PARAMS  = 'kmeans'   # Стратегія стартової ініціалізації (Крок 0 для EM): 'kmeans' - розумний розвідник для надійного старту | 'random' - повністю випадкові координати в просторі | 'random_from_data' - випадкові реальні точки з набору даних
SCALER_TYPE      = 'std'      # Алгоритм масштабування простору ознак: 'std' - центрує дисперсію навколо нуля (ідеально для GMM) | 'minmax' - жорстко стискає дані в межі від 0 до 1 | 'norm' - нормує самі вектори по їхній абсолютній довжині
N_INIT           = 15         # Кількість перезапусків EM-алгоритму: захист від застрягання моделі в поганих локальних мінімумах
RANDOM_STATE     = 42         # Фіксація генератора псевдовипадкових чисел: гарантує 100% відтворюваність результатів експерименту

MODEL_DIR        = "GMM_Models"    # Папка для збереження серіалізованих об'єктів
MODEL_PATH       = os.path.join(MODEL_DIR, f"gmm_{TARGET_YEAR}_{COVARIANCE_TYPE}_{GMM_INIT_PARAMS}_model.pkl") # Динамічне ім'я моделі
SCALER_PATH      = os.path.join(MODEL_DIR, f"scaler_{TARGET_YEAR}_{SCALER_TYPE}.pkl")                          # Динамічне ім'я скейлера

# 4. ВІЗУАЛІЗАЦІЯ ТА UI
PLOT_TEMPLATE           = "plotly_dark"                         # Темна тема для інтерактивних графіків Plotly
MAP_LOCATION_MODE       = "country names"                       # Режим розпізнавання країн для мап Choropleth
COLOR_SCALE_HAPPINESS   = "Viridis"                             # Безперервний градієнт для оригінального індексу щастя
COLOR_PALETTE_FULL      = ['#ff4d4d', '#ffcc00', '#00cc66']     # Контрастні семантичні кольори для 3-х кластерів (Червоний, Жовтий, Зелений)
COLOR_PALETTE_MINI      = px.colors.qualitative.Pastel          # Пастельні кольори для експерименту зі зменшеною розмірністю

CLUSTER_LABELS          = {0: "Низький рівень", 1: "Середній рівень", 2: "Високий рівень"}         # Бізнес-назви наших 3-х кластерів
TABLE_PROPS             = {'background-color': '#1e1e1e', 'color': '#00c3ff', 'border': '1px solid #444', 'text-align': 'center'}
DESCRIBE_CMAP           = 'YlGn'                                # Кольорова схема (Yellow-Green) для підсвічування описових статистик
FEATURE_TRANSLATIONS = { 										# Переклад соціально-економічних ознак
    "Logged GDP per capita": "ВВП на душу населення",
    "Economy (GDP per Capita)": "ВВП на душу населення",
    "Economy..GDP.per.Capita.": "ВВП на душу населення",

    "Social support": "Соціальна підтримка",
    "Family": "Сім'я (Соціальна підтримка)",

    "Healthy life expectancy": "Тривалість здорового життя",
    "Health (Life Expectancy)": "Тривалість здорового життя",
    "Health..Life.Expectancy.": "Тривалість здорового життя",

    "Freedom to make life choices": "Свобода вибору",
    "Freedom": "Свобода вибору",

    "Perceptions of corruption": "Сприйняття корупції",
    "Trust (Government Corruption)": "Сприйняття корупції",
    "Trust..Government.Corruption.": "Сприйняття корупції"
}

print("⚙️ Глобальні константи ініціалізовано!")
print(f"   📰 Рік: {TARGET_YEAR}")
print(f"   🔗 Джерело: {KAGGLE_SOURCES[TARGET_YEAR]}")
print(f"   📊 Ознаки: {len(FEATURES_FULL)} вимірів")
print(f"   🤖 Машинне навчання: GMM({COVARIANCE_TYPE}, {GMM_INIT_PARAMS}) + {SCALER_TYPE} Scaler")

⚙️ Глобальні константи ініціалізовано!
   📰 Рік: 2023
   🔗 Джерело: ajaypalsinghlo/world-happiness-report-2023
   📊 Ознаки: 5 вимірів
   🤖 Машинне навчання: GMM(full, kmeans) + std Scaler


**1.7. Приклад на HTML (Анатомія GMM):**

In [5]:
C_RAW = "#888888"                               # Базовий колір для "нерозмічених" (сирих) даних у просторі
C_C1  = COLOR_PALETTE_FULL[0]                   # Динамічний колір Кластера 1 (підтягується з глобальної палітри констант)
C_C2  = COLOR_PALETTE_FULL[1]                   # Динамічний колір Кластера 2
C_C3  = COLOR_PALETTE_FULL[2]                   # Динамічний колір Кластера 3

html_em_pipeline = f"""
<div style="font-family: sans-serif; max-width: 900px; background-color: #111; padding: 20px; border-radius: 10px; border: 1px solid #333; margin: auto;">
    <h2 style="color: #00c3ff; text-align: center; margin-top: 0;">🧠 Анатомія GMM: Що робить EM-алгоритм з країнами?</h2>
    
    <div style="background-color: #1a1a1a; padding: 15px; margin-bottom: 15px; border-left: 5px solid {C_RAW}; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 0: Сирий простір (Дані після {SCALER_TYPE} Scaler)</div>
        <div style="color: {C_RAW}; font-size: 15px; margin-top: 5px; font-style: italic;">
            Маємо N країн у багатовимірному просторі ознак (ВВП, Здоров'я, Свобода...).<br>
            Усі точки "сірі", алгоритм ще нічого не знає про кластери.
        </div>
    </div>

    <div style="text-align: center; color: #ffd700; font-size: 20px;">⬇</div>

    <div style="background-color: #1a1a1a; padding: 15px; margin: 15px 0; border-left: 5px solid #ffd700; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 1: Ініціалізація (Метод '{GMM_INIT_PARAMS}')</div>
        <div style="color: #ffd700; font-size: 15px; margin-top: 5px;">
            ШІ генерує {N_CLUSTERS} випадкові багатовимірні "дзвони" (Гаусівські розподіли).<br>
            Кожен має свій центр <b>(μ)</b> та матрицю коваріації <b>(Σ)</b>.
        </div>
    </div>

    <div style="text-align: center; color: #ff9900; font-size: 20px;">⬇ ♻️ Цикл EM-алгоритму ♻️ ⬇</div>

    <div style="background-color: #1a1a1a; padding: 15px; margin: 15px 0; border-left: 5px solid #ff9900; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 2: E-крок (Expectation / Очікування)</div>
        <div style="color: #ff9900; font-size: 15px; margin-top: 5px;">
            Обчислення м'якої ймовірності (Soft Clustering) за формулою Баєса:<br>
            <i>"Країна Х належить до Кластера-1 на 10%, Кластера-2 на 85%, Кластера-3 на 5%".</i>
        </div>
    </div>

    <div style="text-align: center; color: #00aaff; font-size: 20px;">⬇</div>

    <div style="background-color: #1a1a1a; padding: 15px; margin: 15px 0; border-left: 5px solid #00aaff; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 3: M-крок (Maximization / Максимізація)</div>
        <div style="color: #00aaff; font-size: 15px; margin-top: 5px;">
            Оновлення параметрів дзвонів: <b>Нові μ</b> тягнуться до скупчень точок, <b>Нові Σ</b> змінюють форму еліпсів.
        </div>
    </div>

    <div style="text-align: center; color: #00ffcc; font-size: 20px; margin-top: 10px;">⬇</div>

    <div style="background-color: #222; padding: 15px; margin-top: 15px; border: 2px dashed #00ffcc; border-radius: 5px; text-align: center;">
        <div style="color: #888; font-size: 14px; font-weight: bold; text-transform: uppercase;">✓ Фінал: Збіжність (Convergence)</div>
        <div style="color: #00ffcc; font-size: 18px; margin-top: 10px; font-family: monospace;">[ <span style="color:{C_C1}">Кластер 1</span> | <span style="color:{C_C2}">Кластер 2</span> | <span style="color:{C_C3}">Кластер 3</span> ]</div>
    </div>
</div>
"""

print("Красивий Вивід (Інтерактивна схема логіки алгоритму):")
display(HTML(html_em_pipeline))

Красивий Вивід (Інтерактивна схема логіки алгоритму):


**2. Завантажити дані:**

In [6]:
def is_valid_zip(filepath):
    if not os.path.exists(filepath) or not zipfile.is_zipfile(filepath):
        return False
    try:
        with zipfile.ZipFile(filepath, 'r') as z:
            if z.testzip() is not None:
                return False
    except Exception:
        return False
    return True

def download_dataset():
    os.makedirs(DATA_DIR, exist_ok=True)
    print(f"⏳ Завантаження архіву у папку '{DATA_DIR}'...")
    try:
        urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)
        print("✅ Завантаження завершено.")
    except Exception as e:
        print(f"❌ Мережева помилка завантаження: {e}")

if os.path.exists(ZIP_PATH):
    print("🔍 Перевірка цілісності існуючого архіву...")
    if not is_valid_zip(ZIP_PATH):
        print("🪫 Архів пошкоджено. Видаляємо та завантажуємо наново...")
        os.remove(ZIP_PATH)
        download_dataset()
    else:
        print("🔋 Архів цілий. Пропускаємо мережевий запит.")
else:
    download_dataset()

🔍 Перевірка цілісності існуючого архіву...
🔋 Архів цілий. Пропускаємо мережевий запит.


**3. Розпакувати дані:**

In [7]:
if os.path.exists(CSV_FILENAME):
    print(f"⚡ Файл '{CSV_FILENAME}' вже розпаковано та готовий до роботи.")
elif os.path.exists(ZIP_PATH):
    if is_valid_zip(ZIP_PATH):
        print(f"📦 Аналізуємо вміст архіву '{ZIP_PATH}'...")
        try:
            with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
                csv_files = [f for f in zip_ref.namelist() if f.endswith('.csv')]
                if not csv_files:
                    raise Exception("В архіві немає CSV файлів!")

                expected_file_suffix = f"{TARGET_YEAR}.csv"
                target_csv_in_zip = next((f for f in csv_files if f.endswith(expected_file_suffix)), None)

                if not target_csv_in_zip:
                    print(f"   ⚠️ Доступні файли в архіві: {csv_files}")
                    raise Exception(f"Файл для {TARGET_YEAR} року не знайдено в архіві!")

                print(f"   🎯 Знайдено цільовий файл: '{target_csv_in_zip}'")

                tmp_csv_path = CSV_FILENAME + ".tmp"

                try:
                    print(f"   ⚙️ Витягуємо '{target_csv_in_zip}' атомарно...")
                    with zip_ref.open(target_csv_in_zip) as source, open(tmp_csv_path, "wb") as target:
                        shutil.copyfileobj(source, target)

                    if os.path.exists(CSV_FILENAME):
                        os.remove(CSV_FILENAME)
                    os.rename(tmp_csv_path, CSV_FILENAME)
                    print(f"✅ Успіх! Файл '{CSV_FILENAME}' (дані {TARGET_YEAR} року) збережено безпечно.")

                except PermissionError:
                    raise Exception(f"Файл {CSV_FILENAME} заблоковано іншою програмою. Закрийте Excel або інші скрипти.")
                except Exception as extract_err:
                    raise Exception(f"Помилка фізичного запису на диск: {extract_err}")
                finally:
                    if os.path.exists(tmp_csv_path):
                        os.remove(tmp_csv_path)

        except Exception as e:
            print(f"❌ Системна помилка під час роботи з архівом: {e}")
    else:
        print("❌ Критична помилка: Архів досі пошкоджений.")
else:
    print("❌ Помилка: Архів не знайдено. Перезапустіть попередній блок завантаження.")

⚡ Файл 'WorldHappinessDataSet/2023.csv' вже розпаковано та готовий до роботи.


**4. Прочитати дані та відобразити загальну інформацію:**

In [8]:
print(f"📂 Завантаження набору даних з файлу: {CSV_FILENAME}\n")
df = pd.read_csv(CSV_FILENAME)

print("Красивий Вивід - Перші 5 рядків набору даних:")
display(df.head().style.background_gradient(cmap='Blues').set_properties(**TABLE_PROPS))

print("Технічний Вивід:\nОписові статистики:")
display(df.describe().T.style.background_gradient(cmap=DESCRIBE_CMAP).format("{:.4f}"))

print("Інформація про типи ознак та пропуски:")
df.info()

📂 Завантаження набору даних з файлу: WorldHappinessDataSet/2023.csv

Красивий Вивід - Перші 5 рядків набору даних:


,Country name,Ladder score,Standard error of ladder score,upperwhisker,lowerwhisker,Logged GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption,Ladder score in Dystopia,Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,Explained by: Generosity,Explained by: Perceptions of corruption,Dystopia + residual
0,Finland,7.804000,0.036000,7.875000,7.733000,10.792000,0.969000,71.150000,0.961000,-0.019000,0.182000,1.778000,1.888000,1.585000,0.535000,0.772000,0.126000,0.535000,2.363000
1,Denmark,7.586000,0.041000,7.667000,7.506000,10.962000,0.954000,71.250000,0.934000,0.134000,0.196000,1.778000,1.949000,1.548000,0.537000,0.734000,0.208000,0.525000,2.084000
2,Iceland,7.530000,0.049000,7.625000,7.434000,10.896000,0.983000,72.050000,0.936000,0.211000,0.668000,1.778000,1.926000,1.620000,0.559000,0.738000,0.250000,0.187000,2.250000
3,Israel,7.473000,0.032000,7.535000,7.411000,10.639000,0.943000,72.697000,0.809000,-0.023000,0.708000,1.778000,1.833000,1.521000,0.577000,0.569000,0.124000,0.158000,2.691000
4,Netherlands,7.403000,0.029000,7.460000,7.346000,10.942000,0.930000,71.550000,0.887000,0.213000,0.379000,1.778000,1.942000,1.488000,0.545000,0.672000,0.251000,0.394000,2.110000


Технічний Вивід:
Описові статистики:


,count,mean,std,min,25%,50%,75%,max
Ladder score,137.0000,5.5398,1.1399,1.8590,4.7240,5.6840,6.3340,7.8040
Standard error of ladder score,137.0000,0.0647,0.0230,0.0290,0.0470,0.0600,0.0770,0.1470
upperwhisker,137.0000,5.6665,1.1174,1.9230,4.9800,5.7970,6.4410,7.8750
lowerwhisker,137.0000,5.4130,1.1637,1.7950,4.4960,5.5290,6.2430,7.7330
Logged GDP per capita,137.0000,9.4498,1.2073,5.5270,8.5910,9.5670,10.5400,11.6600
Social support,137.0000,0.7991,0.1292,0.3410,0.7220,0.8270,0.8960,0.9830
Healthy life expectancy,136.0000,64.9676,5.7504,51.5300,60.6485,65.8375,69.4125,77.2800
Freedom to make life choices,137.0000,0.7874,0.1124,0.3820,0.7240,0.8010,0.8740,0.9610
Generosity,137.0000,0.0224,0.1417,-0.2540,-0.0740,0.0010,0.1170,0.5310
Perceptions of corruption,137.0000,0.7254,0.1770,0.1460,0.6680,0.7740,0.8460,0.9290


Інформація про типи ознак та пропуски:
<class 'pandas.DataFrame'>
RangeIndex: 137 entries, 0 to 136
Data columns (total 19 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   Country name                                137 non-null    str    
 1   Ladder score                                137 non-null    float64
 2   Standard error of ladder score              137 non-null    float64
 3   upperwhisker                                137 non-null    float64
 4   lowerwhisker                                137 non-null    float64
 5   Logged GDP per capita                       137 non-null    float64
 6   Social support                              137 non-null    float64
 7   Healthy life expectancy                     136 non-null    float64
 8   Freedom to make life choices                137 non-null    float64
 9   Generosity                                  137 non-null   

**5. Побудувати діаграми розподілу числових ознак:**

In [32]:
raw_features = [TARGET_METRIC] + FEATURES_FULL
features_to_plot = [f for f in raw_features if f in df.columns]

num_plots = len(features_to_plot)
cols = 3
rows = math.ceil(num_plots / cols)

clean_titles = [f"<b>{str(f).replace('.', ' ').replace('_', ' ').strip()}</b>" for f in features_to_plot]

fig_dist = make_subplots(
    rows=rows, cols=cols, 
    subplot_titles=clean_titles,
    vertical_spacing=0.18,
    horizontal_spacing=0.08
)

for i, feature in enumerate(features_to_plot):
    row = (i // cols) + 1
    col = (i % cols) + 1
    data = df[feature].dropna()

    fig_dist.add_trace(
        go.Histogram(
            x=data, histnorm='probability density', 
            name=f"{feature}", marker_color='#00c3ff', opacity=0.6, nbinsx=25,
            hovertemplate="<b>Діапазон значень:</b> %{x}<br><b>Емпірична щільність:</b> %{y:.4f}<extra></extra>"
        ), row=row, col=col
    )

    mu, std = data.mean(), data.std()
    x_curve = np.linspace(data.min(), data.max(), 100)
    y_curve = stats.norm.pdf(x_curve, mu, std)

    fig_dist.add_trace(
        go.Scatter(
            x=x_curve, y=y_curve, mode='lines', 
            name=f"Ідеальний Гаусс", line=dict(color='#ffd700', width=3, dash='dot'),
            hovertemplate="<b>Ідеальний Гаусс</b><br>Значення ознаки: %{x:.2f}<br>Теоретична щільність: %{y:.4f}<extra></extra>"
        ), row=row, col=col
    )

    fig_dist.update_xaxes(title_text="Значення", title_font=dict(size=11, color="#888"), showgrid=True, gridcolor='#333', row=row, col=col)
    fig_dist.update_yaxes(title_text="Щільність", title_font=dict(size=11, color="#888"), showgrid=True, gridcolor='#333', row=row, col=col)

fig_dist.update_layout(
    height=350 * rows + 80, 
    width=1500, 
    title_text="📊 Перевірка на нормальність: Реальний розподіл vs Ідеальний Дзвін Гаусса", 
    title_x=0.5,
    template=PLOT_TEMPLATE,
    showlegend=False,
    hovermode="x unified",
    margin=dict(b=100),
    annotations=[
        dict(
            x=0.5, y=-0.075, xref="paper", yref="paper",
            text="🟡 <b>Жовтий пунктир</b> — ідеальна математична модель (Дзвін Гаусса).<br>🟦 <b>Блакитні стовпці</b> — реальний емпіричний розподіл ознаки в наборі даних.",
            showarrow=False, font=dict(size=14, color="#cccccc"), align="center", xanchor="center", yanchor="top"
        )
    ]
)

print("Красивий Вивід:")
fig_dist.show()

Красивий Вивід:


**5.1. Висновок до Кроку 5:**

### Аналіз нормальності розподілу

Для кожної ознаки ми побудували ідеальну теоретичну криву (жовтий пунктир), яка описується рівнянням щільності одномірного нормального розподілу:

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{1}{2}\left(\frac{x-\mu}{\sigma}\right)^2}$$

де $\mu$ — математичне сподівання (середнє значення), а $\sigma$ — стандартне відхилення ознаки. Порівнюючи емпіричні гістограми з цією теоретичною моделлю, робимо такі висновки щодо природи соціально-економічних даних:

1. **Відсутність ідеальної нормальності:** Більшість ознак у наборі даних **не мають** ідеально симетричного нормального розподілу. Реальні макроекономічні дані схильні до перекосів (Skewness) та нетипових викидів.
2. **Лівостороння асиметрія (Negative Skew):** Ознаки `Economy (GDP)` та `Health (Life Expectancy)` помітно зміщені вправо. Це означає, що у світі переважають країни із середнім та високим рівнем життя, тоді як країни з абсолютною бідністю формують довгий, але тонкий лівий "хвіст".
3. **Правостороння асиметрія (Positive Skew):** Ознака `Trust (Government Corruption)` має яскраво виражений експоненційний спад. Переважна більшість країн має дуже низький індекс довіри до уряду, і лише одиничні геополітичні "аномалії" (як-от країни Скандинавії) мають високі показники, утворюючи правий "хвіст".
4. **Вплив на GMM-кластеризацію:** Жорсткий алгоритм K-Means працює погано з такими асиметричними даними, оскільки намагається вписати точки в ідеальні сфери. Натомість `GaussianMixture` (Модель суміші Гаусів) чудово впорається з цим завданням з двох причин:
    - Використання повної матриці коваріації (`covariance_type='full'`) дозволяє кластерам набувати форми витягнутих еліпсів.
    - Математично доведено, що лінійна комбінація кількох Гаусівських розподілів здатна апроксимувати будь-який, навіть найскладніший і найбільш асиметричний розподіл даних.

**6. Відібрати числових ознак та кореляційну матрицю:**

In [33]:
print("🔍 Відбір числових ознак для аналізу...")

selected_columns = [TARGET_METRIC] + [f for f in FEATURES_FULL if f in df.columns]
df_selected = df[selected_columns].copy()

df_numeric = df_selected.select_dtypes(include=[np.number])

print(f"✅ Відібрано числових ознак: {len(df_numeric.columns)} (включно з цільовою метрикою).")
print("📊 Побудова матриці кореляцій...")

corr_matrix = df_numeric.corr()

clean_labels = [FEATURE_TRANSLATIONS.get(col, str(col).replace('.', ' ').strip()) for col in corr_matrix.columns]

fig_corr = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=clean_labels,
    y=clean_labels,
    colorscale='RdBu_r',
    zmin=-1, zmax=1,
    text=np.round(corr_matrix.values, 2),
    texttemplate="%{text}",
    textfont={"size": 13, "color": "white"},
    hoverinfo="text",
    hovertemplate="<b>Ознака X:</b> %{x}<br><b>Ознака Y:</b> %{y}<br><b>Кореляція:</b> %{z:.4f}<extra></extra>"
))

fig_corr.update_layout(
    title_text="🔥 Матриця кореляцій Пірсона (Взаємозв'язок факторів)",
    title_x=0.5,
    width=750,
    height=750,
    template=PLOT_TEMPLATE,
    xaxis=dict(tickangle=-45),
    margin=dict(b=120)
)

print("\nКрасивий Вивід:")
fig_corr.show()

🔍 Відбір числових ознак для аналізу...
✅ Відібрано числових ознак: 6 (включно з цільовою метрикою).
📊 Побудова матриці кореляцій...

Красивий Вивід:


**7. Зробити висновок:**

### Аналіз наявності та сили лінійного зв'язку

Для оцінки взаємозв'язків ми побудували теплову карту на основі **коефіцієнта кореляції Пірсона ($r$)**, який обчислюється за формулою:

$$r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}}$$

де $\bar{x}$ та $\bar{y}$ — середні значення відповідних ознак. Цей коефіцієнт варіюється в межах $[-1; 1]$, де $1$ — ідеальна пряма залежність, $0$ — відсутність лінійного зв'язку, а $-1$ — ідеальна зворотна залежність. 

Аналізуючи отриману матрицю, можна зробити такі висновки щодо відібраних числових ознак:

1. **Сильний позитивний зв'язок із цільовою метрикою:** Найбільший вплив на Індекс щастя (`Happiness Score`) мають економічний фактор `Economy (GDP per Capita)` та соціально-медичний фактор `Health (Life Expectancy)`. Коефіцієнт кореляції для них перевищує 0.75, що вказує на виражену пряму лінійну залежність.
2. **Внутрішня мультиколінеарність (Зв'язок між ознаками):** Спостерігається дуже сильний взаємозв'язок (кореляція ~0.8) між самим ВВП та тривалістю життя. Це логічно з точки зору домену (багатші країни мають кращу медицину). 
3. **Слабкі та помірні зв'язки:** Ознака `Trust (Government Corruption)` демонструє значно слабший зв'язок як з індексом щастя, так і з іншими метриками (кореляція в діапазоні 0.2 - 0.4). Це свідчить про те, що цей фактор є більш незалежним і додає системі унікальної дисперсії.
4. **Вплив на подальшу кластеризацію:** Наявність сильної мультиколінеарності означає, що хмара даних у багатовимірному просторі має форму витягнутого еліпсоїда, а не ідеальної сфери. Саме тому використання алгоритму Gaussian Mixture Model (GMM) з параметром `covariance_type='full'` є архітектурно правильним рішенням — цей алгоритм здатен адаптуватися до таких лінійних зв'язків та коректно розділити простір.

**8. Відобразити розподіл:**

In [15]:
print(f"🌍 Побудова теплової мапи для цільової ознаки '{TARGET_METRIC}'...\n")

print("🏗️ Перетворення назв країн у ISO-3 коди...")

MANUAL_ISO_MAPPING = {
    "Turkiye": "TUR",
    "Taiwan Province of China": "TWN",
    "Hong Kong S.A.R. of China": "HKG",
    "State of Palestine": "PSE",
    "Congo (Brazzaville)": "COG",
    "Congo (Kinshasa)": "COD",
    "Kosovo": "XKX",
    "Czechia": "CZE",
    "Ivory Coast": "CIV"
}

def get_iso3_code(country_name):
    if country_name in MANUAL_ISO_MAPPING:
        return MANUAL_ISO_MAPPING[country_name]
    try:
        return pycountry.countries.search_fuzzy(country_name)[0].alpha_3
    except:
        return None

df['ISO_Code'] = df[COUNTRY_COL].apply(get_iso3_code)

missing_countries = df[df['ISO_Code'].isna()][COUNTRY_COL].unique()
if len(missing_countries) > 0:
    print(f"⚠️ Увага! Додайте ці країни у MANUAL_ISO_MAPPING: {missing_countries}")
else:
    print("✅ Усі країни успішно перетворено в ISO-3!")

fig_original_map = px.choropleth(
    df,
    locations='ISO_Code',            
    color=TARGET_METRIC,               
    locationmode='ISO-3',            
    color_continuous_scale=COLOR_SCALE_HAPPINESS, 
    hover_name=COUNTRY_COL,
    labels={TARGET_METRIC: "Індекс щастя", 'ISO_Code': "Код ISO"} 
)

fig_original_map.update_traces(
    hovertemplate="<b>%{hovertext}</b><br><br>" +
                  "📌 Код країни: <b>%{location}</b><br>" +
                  "📊 Рівень щастя: <b>%{z:.3f}</b><br>" +
                  "<extra></extra>"
)

fig_original_map.update_layout(
    title_text=f"🗺️ Happiness Index {TARGET_YEAR} (Оригінальні дані)",
    title_x=0.5,
    template=PLOT_TEMPLATE,            
    geo=dict(
        showframe=False,
        showcoastlines=True, coastlinecolor="rgba(255, 255, 255, 0.2)",
        projection_type='natural earth',
        bgcolor='rgba(0,0,0,0)',
        lakecolor='#111111',
        landcolor='#222222'
    ),
    width=1200, height=700
)

print("\nКрасивий Вивід - Оригінальна мапа щастя:")
fig_original_map.show()

print("Технічний Вивід - Екстремуми рейтингу (ТОП-5 та Анти-ТОП-5 країн):")
top_bottom_df = pd.concat([
    df[[COUNTRY_COL, TARGET_METRIC]].nlargest(5, TARGET_METRIC),
    df[[COUNTRY_COL, TARGET_METRIC]].nsmallest(5, TARGET_METRIC).sort_values(by=TARGET_METRIC, ascending=False)
])

display(top_bottom_df.style.background_gradient(cmap=DESCRIBE_CMAP, subset=[TARGET_METRIC])\
        .set_properties(**TABLE_PROPS).format({TARGET_METRIC: "{:.4f}"}))

🌍 Побудова теплової мапи для цільової ознаки 'Ladder score'...

🏗️ Перетворення назв країн у ISO-3 коди...
✅ Усі країни успішно перетворено в ISO-3!

Красивий Вивід - Оригінальна мапа щастя:


Технічний Вивід - Екстремуми рейтингу (ТОП-5 та Анти-ТОП-5 країн):


,Country name,Ladder score
0,Finland,7.8040
1,Denmark,7.5860
2,Iceland,7.5300
3,Israel,7.4730
4,Netherlands,7.4030
132,Congo (Kinshasa),3.2070
133,Zimbabwe,3.2040
134,Sierra Leone,3.1380
135,Lebanon,2.3920
136,Afghanistan,1.8590


**9. Застосувати стандартизацію даних:**

In [34]:
print(f"⚖️ Масштабування ознак через функцію data_scale() (Метод: {SCALER_TYPE.upper()})...")

def data_scale(data, scaler_type='minmax'):
    if scaler_type == 'minmax':
        scaler = MinMaxScaler()
    elif scaler_type == 'std':
        scaler = StandardScaler()
    elif scaler_type == 'norm':
        scaler = Normalizer()
    else:
        raise ValueError("Невідомий тип скейлера!")

    scaler.fit(data)
    res = scaler.transform(data)

    return res, scaler

X_features = df[FEATURES_FULL].copy()
data_scaled, fitted_scaler = data_scale(X_features, scaler_type=SCALER_TYPE)
df_scaled = pd.DataFrame(data_scaled, columns=FEATURES_FULL, index=df.index)

print("✅ Масштабування завершено успішно! (Скейлер збережено в оперативній пам'яті)\n")

f1, f2 = FEATURES_FULL[0], FEATURES_FULL[1]

f1_ua = FEATURE_TRANSLATIONS.get(f1, str(f1))
f2_ua = FEATURE_TRANSLATIONS.get(f2, str(f2))

SCALER_NAMES = {
    'minmax': 'MinMaxScaler (від 0 до 1)',
    'std': 'StandardScaler (Z-score центрування)',
    'norm': 'Normalizer (Векторна нормалізація)'
}
scaler_display_name = SCALER_NAMES.get(SCALER_TYPE, SCALER_TYPE.upper())

fig_scale = make_subplots(
    rows=1, cols=2, 
    subplot_titles=(f"Оригінальні дані", f"Відмасштабовано: {scaler_display_name}"),
    horizontal_spacing=0.1
)

fig_scale.add_trace(go.Scatter(
    x=df[f1], y=df[f2], mode='markers',
    marker=dict(color='#00c3ff', size=9, opacity=0.7, line=dict(width=1, color='black')),
    text=df[COUNTRY_COL], 
    hovertemplate="<b>%{text}</b><br>" + 
                  f"{f1_ua}: <b>%{{x:.3f}}</b><br>" + 
                  f"{f2_ua}: <b>%{{y:.3f}}</b><extra></extra>"
), row=1, col=1)

fig_scale.add_trace(go.Scatter(
    x=df_scaled[f1], y=df_scaled[f2], mode='markers',
    marker=dict(color='#ccff00', size=9, opacity=0.7, line=dict(width=1, color='black')),
    text=df[COUNTRY_COL], 
    hovertemplate="<b>%{text}</b><br>" + 
                  f"Масштаб. {f1_ua}: <b>%{{x:.3f}}</b><br>" + 
                  f"Масштаб. {f2_ua}: <b>%{{y:.3f}}</b><extra></extra>"
), row=1, col=2)

fig_scale.update_xaxes(title_text=f1_ua, showgrid=True, gridcolor='#333', row=1, col=1)
fig_scale.update_yaxes(title_text=f2_ua, showgrid=True, gridcolor='#333', row=1, col=1)

fig_scale.update_xaxes(title_text=f"Scaled: {f1_ua}", showgrid=True, gridcolor='#333', row=1, col=2)
fig_scale.update_yaxes(title_text=f"Scaled: {f2_ua}", showgrid=True, gridcolor='#333', row=1, col=2)

fig_scale.update_layout(
    title_text="✨ Трансформація простору: Оригінальні vs Відмасштабовані дані",
    title_x=0.5, width=1500, height=550, template=PLOT_TEMPLATE, showlegend=False,
    margin=dict(b=80)
)

fig_scale.add_annotation(
    x=0.5, y=-0.20, xref="paper", yref="paper",
    text="💡 Зверніть увагу на осі: форма хмари точок зберігається, але координати стиснуті алгоритмом для потреб машинного навчання.",
    showarrow=False, font=dict(size=14, color="#cccccc"), align="center"
)

print("Красивий Вивід - Візуалізація ефекту масштабування:")
fig_scale.show()

print("\nТехнічний Вивід - Перші 5 рядків відмасштабованих ознак:")
display(df_scaled.head().style.background_gradient(cmap='Purples').set_properties(**TABLE_PROPS).format("{:.4f}"))

⚖️ Масштабування ознак через функцію data_scale() (Метод: STD)...
✅ Масштабування завершено успішно! (Скейлер збережено в оперативній пам'яті)

Красивий Вивід - Візуалізація ефекту масштабування:



Технічний Вивід - Перші 5 рядків відмасштабованих ознак:


,Logged GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Perceptions of corruption
0,1.1158,1.3198,1.0791,1.5506,-3.0821
1,1.2571,1.2033,1.0966,1.3094,-3.0027
2,1.2023,1.4286,1.2362,1.3273,-0.3256
3,0.9886,1.1179,1.3491,0.1930,-0.0987
4,1.2405,1.0169,1.1489,0.8897,-1.9647


**10. Відобразити статистики:**

In [35]:
print("📊 Технічний аудит та порівняння описових статистик...\n")

fig_stats = make_subplots(
    rows=2, cols=1,
    subplot_titles=("1. Розподіл ОРИГІНАЛЬНИХ ознак (Різні масштаби та дисперсії)", f"2. Розподіл ВІДМАСШТАБОВАНИХ ознак ({SCALER_TYPE.upper()})"),
    vertical_spacing=0.12
)

colors = px.colors.qualitative.Pastel

for i, col in enumerate(FEATURES_FULL):
    clean_name = FEATURE_TRANSLATIONS.get(col, str(col).replace('.', ' ').strip())

    fig_stats.add_trace(go.Violin(
        x=X_features[col], name=clean_name, marker_color=colors[i % len(colors)],
        box_visible=True, meanline_visible=True, points='outliers',
        hovertemplate=f"<b>{clean_name}</b><br>Значення: %{{x:.3f}}<extra></extra>"
    ), row=1, col=1)

for i, col in enumerate(FEATURES_FULL):
    clean_name = FEATURE_TRANSLATIONS.get(col, str(col).replace('.', ' ').strip())
    
    fig_stats.add_trace(go.Violin(
        x=df_scaled[col], name=clean_name, marker_color=colors[i % len(colors)],
        box_visible=True, meanline_visible=True, points='outliers',
        hovertemplate=f"<b>{clean_name} (Scaled)</b><br>Значення: %{{x:.3f}}<extra></extra>"
    ), row=2, col=1)

fig_stats.update_xaxes(title_text="Оригінальні значення", showgrid=True, gridcolor='#333', row=1, col=1)
fig_stats.update_yaxes(title_text="Ознаки", showgrid=True, gridcolor='#333', row=1, col=1)

fig_stats.update_xaxes(title_text="Відмасштабовані значення", showgrid=True, gridcolor='#333', row=2, col=1)
fig_stats.update_yaxes(title_text="Ознаки", showgrid=True, gridcolor='#333', row=2, col=1)

fig_stats.update_layout(
    title_text="🎻 'Скрипкові діаграми' (Violin Plots): Аналіз щільності та квартилів",
    title_x=0.5, 
    width=1500, height=950,
    template=PLOT_TEMPLATE, showlegend=False,
    margin=dict(b=130, l=150, t=80)
)

fig_stats.add_annotation(
    x=0.5, y=-0.13, xref="paper", yref="paper",
    text="💡 <b>Violin Plot</b> поєднує Boxplot (всередині) та хвилю щільності (зовні). Товщина 'скрипки' показує, де сконцентровано найбільше країн.<br>Зверніть увагу, як нижній графік вирівняв дисперсію всіх ознак, підготувавши їх до GMM!",
    showarrow=False, font=dict(size=14, color="#cccccc"), align="center"
)

print("Красивий Вивід:")
fig_stats.show()

print("Технічний Вивід - Статистики ОРИГІНАЛЬНОГО набору (Для порівняння):")
display(X_features.describe().T.style.background_gradient(cmap=DESCRIBE_CMAP).format("{:.4f}"))

print("\nТехнічний Вивід - Статистики ВІДМАСШТАБОВАНОГО набору (Scaled):")
display(df_scaled.describe().T.style.background_gradient(cmap='Purples').format("{:.4f}"))

📊 Технічний аудит та порівняння описових статистик...

Красивий Вивід:


Технічний Вивід - Статистики ОРИГІНАЛЬНОГО набору (Для порівняння):


,count,mean,std,min,25%,50%,75%,max
Logged GDP per capita,137.0000,9.4498,1.2073,5.5270,8.5910,9.5670,10.5400,11.6600
Social support,137.0000,0.7991,0.1292,0.3410,0.7220,0.8270,0.8960,0.9830
Healthy life expectancy,136.0000,64.9676,5.7504,51.5300,60.6485,65.8375,69.4125,77.2800
Freedom to make life choices,137.0000,0.7874,0.1124,0.3820,0.7240,0.8010,0.8740,0.9610
Perceptions of corruption,137.0000,0.7254,0.1770,0.1460,0.6680,0.7740,0.8460,0.9290



Технічний Вивід - Статистики ВІДМАСШТАБОВАНОГО набору (Scaled):


,count,mean,std,min,25%,50%,75%,max
Logged GDP per capita,137.0000,0.0000,1.0037,-3.2611,-0.7139,0.0974,0.9063,1.8374
Social support,137.0000,-0.0000,1.0037,-3.5579,-0.5986,0.2169,0.7528,1.4286
Healthy life expectancy,136.0000,-0.0000,1.0037,-2.3455,-0.7539,0.1518,0.7758,2.1491
Freedom to make life choices,137.0000,0.0000,1.0037,-3.6209,-0.5662,0.1215,0.7735,1.5506
Perceptions of corruption,137.0000,-0.0000,1.0037,-3.2863,-0.3256,0.2756,0.6840,1.1548


**10.1. Висновок до Кроку 10:**

### Аналіз стандартизованих статистик та щільності розподілу

Порівнюючи описові статистики (`describe()`) та їх візуалізацію через Скрипкові діаграми (Violin Plots) для оригінального та відмасштабованого наборів даних, можна зробити такі математичні висновки:

1. **Трансформація простору (Feature Scaling):** В оригінальному наборі ознаки мали кардинально різні математичні діапазони. Після застосування нашого скейлера всі вектори ознак $x$ були лінійно трансформовані у новий простір $x'$. Наприклад, при стандартизації (StandardScaler) це досягається зміщенням середнього до нуля та нормуванням дисперсії до одиниці:
    $$x'_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$
    Це жорстко вписало всі ознаки в єдиний стандартизований масштаб, що яскраво видно по вирівняних "скрипках" на нижньому графіку. 

2. **Уніфікація статистичної ваги:** Завдяки масштабуванню не лише медіани, але й дисперсії (ширина хвилі розподілу) були збалансовані. Це означає, що відтепер жодна ознака не зможе штучно "домінувати" в алгоритмі просто за рахунок більших абсолютних чисел (наприклад, макроекономіка більше не задавить соціальні фактори).

3. **Математична стабільність для GMM:** Скрипкові діаграми наочно показують оцінку щільності (Kernel Density). Наявність "потовщень" вказує на скупчення країн. Алгоритм GMM шукає такі згущення, щоб описати їх багатовимірним нормальним розподілом:
    $$\mathcal{N}(x | \mu, \Sigma) = \frac{1}{\sqrt{(2\pi)^D |\Sigma|}} \exp\left(-\frac{1}{2}(x - \mu)^T \Sigma^{-1} (x - \mu)\right)$$
    Де $\Sigma$ — матриця коваріації, а $D$ — розмірність простору. Якби ми не відмасштабували дані, детермінант $|\Sigma|$ та обернена матриця $\Sigma^{-1}$ були б екстремально спотворені ознакою з найбільшою дисперсією. Масштабування гарантує, що багатовимірні еліпси Гаусса формуватимуться на основі реальної геометрії даних, а не через різницю в одиницях виміру.

**11. Побудувати модель кластеризації:**

In [ ]:
print(f"🧠 Навчання моделі кластеризації (GaussianMixture)...")
print("🧹 Перевірка та очищення даних від пропущених значень (NaN)...")

missing_values_count = df_scaled[FEATURES_FULL].isna().sum().sum()

if missing_values_count > 0:
    print(f"   ⚠️ Знайдено {missing_values_count} пропущених значень у числових ознаках.")
    print("   🛠️ Застосовуємо імп'ютацію (заповнення медіаною)...")

    from sklearn.impute import SimpleImputer
    imputer = SimpleImputer(strategy='median')

    df_scaled[FEATURES_FULL] = imputer.fit_transform(df_scaled[FEATURES_FULL])
    df[FEATURES_FULL] = imputer.fit_transform(df[FEATURES_FULL])

    print("   ✅ Пропущені значення успішно заповнено!")
else:
    print("   ✅ Пропущених значень не знайдено. Дані чисті.")

X_train = df_scaled[FEATURES_FULL]
gmm_brain = GaussianMixture(
    n_components=N_CLUSTERS, covariance_type=COVARIANCE_TYPE, 
    n_init=N_INIT, random_state=RANDOM_STATE
)
cluster_labels = gmm_brain.fit_predict(X_train)

df['Cluster'] = cluster_labels
cluster_means = df.groupby('Cluster')[TARGET_METRIC].mean().sort_values()
cluster_mapping = {old_id: new_id for new_id, old_id in enumerate(cluster_means.index)}
df['Cluster'] = df['Cluster'].map(cluster_mapping)

df['Cluster_Name'] = df['Cluster'].map(CLUSTER_LABELS)

actual_iters = gmm_brain.n_iter_
print(f"✅ Навчання завершено! Алгоритм зійшовся за {actual_iters} ітерацій.")

f1, f2 = FEATURES_FULL[0], FEATURES_FULL[1]
f1_ua = FEATURE_TRANSLATIONS.get(f1, str(f1).replace('.', ' ').strip())
f2_ua = FEATURE_TRANSLATIONS.get(f2, str(f2).replace('.', ' ').strip())

print("\nКрасивий Вивід:")

def get_dynamic_legend(X_coords, x_min, x_max, y_min, y_max):
    x_mid = (x_min + x_max) / 2
    y_mid = (y_min + y_max) / 2

    top_left = np.sum((X_coords[:, 0] <= x_mid) & (X_coords[:, 1] >= y_mid))
    top_right = np.sum((X_coords[:, 0] > x_mid) & (X_coords[:, 1] >= y_mid))
    bottom_left = np.sum((X_coords[:, 0] <= x_mid) & (X_coords[:, 1] < y_mid))
    bottom_right = np.sum((X_coords[:, 0] > x_mid) & (X_coords[:, 1] < y_mid))

    min_points = min(top_left, top_right, bottom_left, bottom_right)

    if min_points == top_left:
        return dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
    elif min_points == top_right:
        return dict(yanchor="top", y=0.99, xanchor="right", x=0.99)
    elif min_points == bottom_left:
        return dict(yanchor="bottom", y=0.01, xanchor="left", x=0.01)
    else:
        return dict(yanchor="bottom", y=0.01, xanchor="right", x=0.99)

print(f"\n🎬 1/3: Генерація динамічної анімації навчання...")
X_anim = df_scaled[[f1, f2]].values 
frames = []
em_steps = actual_iters

x_min = np.floor(X_anim[:, 0].min() / 0.5) * 0.5
x_max = np.ceil(X_anim[:, 0].max() / 0.5) * 0.5
y_min = np.floor(X_anim[:, 1].min() / 0.5) * 0.5
y_max = np.ceil(X_anim[:, 1].max() / 0.5) * 0.5

for i in range(1, em_steps + 1):
    gmm_anim = GaussianMixture(
        n_components=N_CLUSTERS, covariance_type=COVARIANCE_TYPE, 
        max_iter=i, n_init=1, init_params='kmeans', random_state=RANDOM_STATE
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        gmm_anim.fit(X_anim)

    current_means = gmm_anim.means_

    frame_traces = [
        go.Scatter(x=X_anim[:, 0], y=X_anim[:, 1], mode='markers', marker=dict(color='#888', size=6, opacity=0.3), showlegend=False)
    ]

    for k in range(N_CLUSTERS):
        frame_traces.append(go.Scatter(
            x=[current_means[k, 0]], y=[current_means[k, 1]], 
            mode='markers+text', text=[f"Гаусс {k+1}"],
            marker=dict(color=COLOR_PALETTE_FULL[k], size=22, line=dict(width=3, color='white')), showlegend=False))

    frames.append(go.Frame(data=frame_traces, name=str(i)))

sliders = [{
    "pad": {"b": 10, "t": 60}, "len": 0.9, "x": 0.1, "y": 0,
    "currentvalue": {"font": {"size": 16}, "prefix": "Ітерація: ", "visible": True, "xanchor": "right"},
    "steps": [{"args": [[f.name], {"frame": {"duration": 400, "redraw": True}, "mode": "immediate", "transition": {"duration": 200}}],
               "label": str(k+1), "method": "animate"} for k, f in enumerate(frames)]
}]

fig_anim = go.Figure(data=frames[0].data, frames=frames)
fig_anim.update_layout(
    title_text=f"🎬 1. Процес навчання: Динаміка EM-алгоритму (2D Зріз)",
    title_x=0.5, template=PLOT_TEMPLATE, width=1050, height=600,
    xaxis=dict(title=f"📐 Scaled: {f1_ua}", range=[x_min, x_max]),
    yaxis=dict(title=f"🩺 Scaled: {f2_ua}", range=[y_min, y_max]),
    updatemenus=[dict(type="buttons", y=-0.1, x=0.05, buttons=[
        dict(label="▶ Play", method="animate", args=[None, {"frame": {"duration": 400, "redraw": True}, "fromcurrent": True, "transition": {"duration": 200}}]),
        dict(label="⏸ Pause", method="animate", args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}])])],
    sliders=sliders
)

fig_anim.show()

print("\n📐 2/3: Побудова контурних карт коваріації...")
f1_idx, f2_idx = 0, 1

x_range, y_range = np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100)
XX, YY = np.meshgrid(x_range, y_range)
pos = np.dstack((XX, YY))

fig_contours = go.Figure()
fig_contours.add_trace(go.Histogram2dContour(x=X_anim[:, 0], y=X_anim[:, 1], colorscale='Greys', opacity=0.3, showscale=False))

for i in range(N_CLUSTERS):
    mean = gmm_brain.means_[i, [f1_idx, f2_idx]]
    cov = gmm_brain.covariances_[i][[f1_idx, f2_idx], :][:, [f1_idx, f2_idx]]
    rv = multivariate_normal(mean=mean, cov=cov)
    Z = rv.pdf(pos)

    color = COLOR_PALETTE_FULL[i]
    name = CLUSTER_LABELS[i]

    fig_contours.add_trace(go.Contour(
        x=x_range, y=y_range, z=Z, colorscale=[[0, 'rgba(0,0,0,0)'], [1, color]],
        showscale=False, contours=dict(start=0.01, coloring='lines'), line=dict(width=1), hoverinfo='skip'))

for i in range(N_CLUSTERS):
    real_indices = (cluster_labels == cluster_mapping[i])
    real_data = df[real_indices]

    hover_texts = [f"<b>{row[COUNTRY_COL]}</b><br>{f1_ua} (Raw): {row[f1]:.2f}<br>{f2_ua} (Raw): {row[f2]:.2f}" for _, row in real_data.iterrows()]

    fig_contours.add_trace(go.Scatter(
        x=X_anim[cluster_labels == i, 0], y=X_anim[cluster_labels == i, 1],
        mode='markers', name=CLUSTER_LABELS[i], marker=dict(color=COLOR_PALETTE_FULL[i], size=7, opacity=0.8, line=dict(width=1, color='black')),
        hovertext=hover_texts, hoverinfo="text",
    ))

padding_x = (x_max - x_min) * 0.10
padding_y = (y_max - y_min) * 0.10

focus_x0 = min(gmm_brain.means_[0, f1_idx], gmm_brain.means_[1, f1_idx]) - padding_x
focus_x1 = max(gmm_brain.means_[0, f1_idx], gmm_brain.means_[1, f1_idx]) + padding_x
focus_y0 = min(gmm_brain.means_[0, f2_idx], gmm_brain.means_[1, f2_idx]) - padding_y
focus_y1 = max(gmm_brain.means_[0, f2_idx], gmm_brain.means_[1, f2_idx]) + padding_y

fig_contours.add_shape(
    type="rect",
    x0=focus_x0, y0=focus_y0, x1=focus_x1, y1=focus_y1,
    line=dict(color="#00ffcc", width=2, dash="dashdot"),
    fillcolor="rgba(0,0,0,0)"
)

fig_contours.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    name="Фокус коваріації",
    line=dict(color="#00ffcc", width=2, dash="dashdot")
))

fig_contours.update_layout(
    title_text=f"📐 2. Геометрія: Гаусівські контурні карти коваріації (GMM)",
    title_x=0.5, template=PLOT_TEMPLATE, width=1050, height=650,
    xaxis=dict(title=f"Scaled: {f1_ua}", showgrid=False, range=[x_min, x_max]),
    yaxis=dict(title=f"Scaled: {f2_ua}", showgrid=False, range=[y_min, y_max]),
    legend=get_dynamic_legend(X_anim, x_min, x_max, y_min, y_max)
)

fig_contours.show()

print("\n🌌 3/3: Стиснення 5-вимірного простору через PCA...")
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_train)

fig_pca = go.Figure()

pca_x_min = np.floor(X_pca[:, 0].min() / 0.5) * 0.5
pca_x_max = np.ceil(X_pca[:, 0].max() / 0.5) * 0.5
pca_y_min = np.floor(X_pca[:, 1].min() / 0.5) * 0.5
pca_y_max = np.ceil(X_pca[:, 1].max() / 0.5) * 0.5

xx, yy = np.meshgrid(np.linspace(pca_x_min, pca_x_max, 100), np.linspace(pca_y_min, pca_y_max, 100))
pos = np.dstack((xx, yy))

for i in range(N_CLUSTERS):
    cluster_points = X_pca[df['Cluster'] == i]

    pca_mean = np.mean(cluster_points, axis=0)
    pca_cov = np.cov(cluster_points, rowvar=False)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        rv = multivariate_normal(mean=pca_mean, cov=pca_cov)
        Z = rv.pdf(pos)

    color = COLOR_PALETTE_FULL[i]
    name = CLUSTER_LABELS[i]

    fig_pca.add_trace(go.Contour(
        x=np.linspace(pca_x_min, pca_x_max, 100), y=np.linspace(pca_y_min, pca_y_max, 100), z=Z,
        showscale=False, contours=dict(start=0.1, coloring='lines'), line=dict(width=1, color=color), hoverinfo='skip'))

    real_indices = (cluster_labels == cluster_mapping[i])
    real_data = df[real_indices]
    hover_texts_pca = [f"<b>{row[COUNTRY_COL]}</b><br>PCA 1: {X_pca[idx, 0]:.2f}<br>PCA 2: {X_pca[idx, 1]:.2f}" for idx, row in real_data.reset_index(drop=True).iterrows()]

    fig_pca.add_trace(go.Scatter(
        x=cluster_points[:, 0], y=cluster_points[:, 1], mode='markers', name=name,
        marker=dict(color=color, size=9, opacity=0.8, line=dict(width=1, color='black')),
        hovertext=hover_texts_pca, hoverinfo="text"
    ))

variance_explained = sum(pca.explained_variance_ratio_) * 100
fig_pca.update_layout(
    title_text=f"🌌 3. Фінальний результат: 5D-Проєкція (PCA). Збережено {variance_explained:.1f}% інформації",
    title_x=0.5, template=PLOT_TEMPLATE, width=1050, height=700,
    xaxis=dict(title=f"← Головна компонента 1 (Абстрактна вісь) →", showgrid=True, gridcolor='#333', range=[pca_x_min, pca_x_max]),
    yaxis=dict(title=f"← Головна компонента 2 (Абстрактна вісь) →", showgrid=True, gridcolor='#333', range=[pca_y_min, pca_y_max]),
    legend=get_dynamic_legend(X_pca, pca_x_min, pca_x_max, pca_y_min, pca_y_max)
)

fig_pca.show()

🧠 Навчання моделі кластеризації (GaussianMixture)...

🧹 Перевірка та очищення даних від пропущених значень (NaN)...
   ✅ Пропущених значень не знайдено. Дані чисті.
✅ Навчання завершено! Алгоритм зійшовся за 9 ітерацій.

Красивий Вивід:

🎬 1/3: Генерація динамічної анімації навчання...



📐 2/3: Побудова контурних карт коваріації...



🌌 3/3: Стиснення 5-вимірного простору через PCA...


**11.5.\*\* Експорт навченої моделі ШІ:**

In [47]:
print(f"🪬 Експорт моделі (Збереження 'Мозку' GMM для {TARGET_YEAR} року)...\n")

if 'gmm_brain' not in globals() or 'fitted_scaler' not in globals():
    raise SystemExit("\n❌ Критична Помилка Архітектури: Розсинхронізація стану!\n"
                     "   У пам'яті зараз відсутня натренована модель GMM або Скейлер.\n"
                     "   👉 РІШЕННЯ: Запустіть попередній блок 'Навчання моделі' ще раз!")

actual_trained_clusters = gmm_brain.n_components
current_model_id = id(gmm_brain)
last_saved_id = globals().get('_LAST_SAVED_MODEL_ID')
last_saved_time = globals().get('_LAST_SAVED_TIMESTAMP_HUMAN')

model_exists = os.path.exists(MODEL_PATH)

if actual_trained_clusters != N_CLUSTERS:
    print("\n❌ Критична Помилка Архітектури: Розсинхронізація стану!")
    print(f"   Ти змінив константу кластерів на [{N_CLUSTERS}].")
    print(f"   Але в пам'яті зараз висить 'Мозок', натренований на [{actual_trained_clusters}] кластерів.")
    print("   👉 РІШЕННЯ: Запусти блок 'Навчання моделі' ще раз, щоб перенавчити математику, а потім повертайся сюди!")
elif current_model_id == last_saved_id and model_exists:
    print(f"   ♻️ Ця конкретна модель вже була збережена на диск (Час фіксації: {last_saved_time}).")
    print("   💬 Вікно збереження автоматично пропущено, щоб уникнути дублювання.")
else:
    timestamp_now = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    timestamp_human = pd.Timestamp.now().strftime("%d.%m.%Y %H:%M:%S")

    default_save = globals().get('DEFAULT_SAVE_MODEL', True)
    default_action_text = '💚 Зберігати' if default_save else '💛 Не зберігати'

    if model_exists:
        base_msg = f"⚠️ Капсула моделі '{MODEL_PATH}' вже існує. Перезаписати її?"
    else:
        base_msg = "❓ Зберегти цю навчену політику (Мозок) на Диск?"

    prompt_msg = f"   {base_msg} (Так/Ні) [За замовчуванням: {default_action_text}]: "
    user_input = input(prompt_msg).strip().lower()

    if user_input in ['y', 'yes', 'т', 'так']:
        should_save = True
    elif user_input in ['n', 'no', 'н', 'ні']:
        should_save = False
    elif user_input == '': 
        should_save = default_save
        print(f"   ↩️ Натиснуто Enter. Використовуємо значення за замовчуванням: {default_action_text}")
    else:
        should_save = default_save
        print(f"   ❓ Відповідь не розпізнана. Використовуємо значення за замовчуванням: {default_action_text}")

    if should_save:
        try:
            model_dir = os.path.dirname(MODEL_PATH)
            if model_dir:
                os.makedirs(model_dir, exist_ok=True)

            if model_exists:
                try:
                    old_payload = joblib.load(MODEL_PATH)
                    old_year = old_payload.get('target_year', 'Unknown')
                    old_time = str(old_payload.get('timestamp', '000000_000000')).replace(':', '-').replace(' ', '_')

                    backup_name = f"GMM_{old_year}_{old_time}.backup"
                    backup_path = os.path.join(model_dir, backup_name) if model_dir else backup_name

                    shutil.copy2(MODEL_PATH, backup_path)
                    print(f"      📥 Створено резервну копію старої моделі: {backup_name}")
                except Exception as backup_err:
                    print(f"      ⚠️ Не вдалося створити резервну копію (можливо, старий файл пошкоджено): {backup_err}")

            model_payload = {
                "algorithm": "Gaussian Mixture Model (GMM)",
                "brain": gmm_brain,
                "scaler": fitted_scaler,
                "scaler_type": SCALER_TYPE,
                "target_year": TARGET_YEAR,
                "features": FEATURES_FULL,
                "cluster_labels": CLUSTER_LABELS,
                "cluster_mapping": cluster_mapping,
                "hyperparameters": {
                    "n_clusters": N_CLUSTERS,
                    "covariance_type": COVARIANCE_TYPE,
                    "init_params": GMM_INIT_PARAMS,
                    "n_init": N_INIT
                },
                "ui_colors": COLOR_PALETTE_FULL,
                "timestamp": timestamp_now
            }

            joblib.dump(model_payload, MODEL_PATH)

            df.to_csv(EXPORT_CSV_PATH, index=False)

            globals()['_LAST_SAVED_MODEL_ID'] = current_model_id
            globals()['_LAST_SAVED_TIMESTAMP_HUMAN'] = timestamp_human

            file_size_kb = os.path.getsize(MODEL_PATH) / 1024
            csv_size_kb = os.path.getsize(EXPORT_CSV_PATH) / 1024

            print("   🎭 Початок збереження у файл...")
            print(f"      📦 Файл: {MODEL_PATH} (Матриці Коваріації + Скейлер + Метадані)")
            print(f"      🎛 Ознаки: {len(FEATURES_FULL)} вимірів (Масштабування: {SCALER_TYPE})")
            print("      👾 Архітектура моделі:")
            print(f"         🧠 Алгоритм: GMM (Expectation-Maximization)")
            print(f"         🧬 Гаусівських розподілів: {N_CLUSTERS}")
            print(f"         🔬 Тип коваріації: {COVARIANCE_TYPE}")
            print(f"         🔄 Ітерацій до збіжності: {gmm_brain.n_iter_}")
            print(f"      🏷 Класи: {', '.join(list(CLUSTER_LABELS.values()))}")
            print(f"   ✅ Успіх! Капсула 'Мозку' ШІ надійно збережена ({file_size_kb:.2f} KB) о {timestamp_human}")
            print(f"   ✅ Набір даних з мітками експортовано ({csv_size_kb:.2f} KB)")
            print("   🎨 UI-Палітра та Словники: Успішно запаковані всередину файлу для майбутнього Інференсу!")

        except Exception as e:
            print(f"\n   ❌ Сталася помилка під час збереження: {e}")
    else:
        print("   ⏭️ Збереження пропущено. Файли на диску не змінено...")

🪬 Експорт моделі (Збереження 'Мозку' GMM для 2023 року)...

   ↩️ Натиснуто Enter. Використовуємо значення за замовчуванням: 💚 Зберігати
   🎭 Початок збереження у файл...
      📦 Файл: GMM_Models/gmm_2023_full_kmeans_model.pkl (Матриці Коваріації + Скейлер + Метадані)
      🎛 Ознаки: 5 вимірів (Масштабування: std)
      👾 Архітектура моделі:
         🧠 Алгоритм: GMM (Expectation-Maximization)
         🧬 Гаусівських розподілів: 3
         🔬 Тип коваріації: full
         🔄 Ітерацій до збіжності: 9
      🏷 Класи: Низький рівень, Середній рівень, Високий рівень
   ✅ Успіх! Капсула 'Мозку' ШІ надійно збережена (4.72 KB) о 29.03.2026 22:00:03
   ✅ Набір даних з мітками експортовано (20.78 KB)
   🎨 UI-Палітра та Словники: Успішно запаковані всередину файлу для майбутнього Інференсу!


**12. Побудувати теплову мапу:**

In [51]:
print("🗺️ Ініціалізація картографічного модуля (Choropleth Map)...")

if 'ISO_Code' not in df.columns or 'Cluster_Name' not in df.columns:
    raise SystemExit("❌ Критична помилка: У датасеті відсутні колонки 'ISO_Code' або 'Cluster_Name'.\n"
                     "   👉 РІШЕННЯ: Переконайтеся, що ви виконали блоки генерації ISO-кодів та навчання GMM.")

print("   ✅ Геодані та мітки кластерів знайдено. Будуємо проєкцію...\n")

# Рахуємо світовий рейтинг (1 - найщасливіша країна)
df['Global_Rank'] = df[TARGET_METRIC].rank(ascending=False, method='min').astype(int)

discrete_color_map = {CLUSTER_LABELS[i]: COLOR_PALETTE_FULL[i] for i in range(N_CLUSTERS)}

f1, f2 = FEATURES_FULL[0], FEATURES_FULL[1]
f1_ua = FEATURE_TRANSLATIONS.get(f1, str(f1).replace('.', ' ').strip())
f2_ua = FEATURE_TRANSLATIONS.get(f2, str(f2).replace('.', ' ').strip())

# 🛡️ ПРАВИЛЬНИЙ ПІДХІД: Передаємо custom_data прямо в px.choropleth
fig_cluster_map = px.choropleth(
    df,
    locations='ISO_Code',            
    color='Cluster_Name',              
    locationmode='ISO-3',            
    color_discrete_map=discrete_color_map,
    hover_name=COUNTRY_COL,
    labels={'Cluster_Name': "Рівень життя (ШІ)"},
    custom_data=['Cluster_Name', TARGET_METRIC, f1, f2, 'Global_Rank'] # <--- ТЕПЕР ВОНО ТУТ
)

# Тепер ми просто форматуємо текст, а дані Plotly підтягне правильно відсортованими
fig_cluster_map.update_traces(
    hovertemplate="<b>%{hovertext}</b><br><br>" +
                  "🏆 Світовий рейтинг: <b>#%{customdata[4]}</b><br>" +
                  "📌 Код ISO: <b>%{location}</b><br>" +
                  "🤖 Кластер ШІ: <b>%{customdata[0]}</b><br>" +
                  "📊 Оригінальний індекс: <b>%{customdata[1]:.3f}</b><br>" +
                  f"💰 {f1_ua}: <b>%{{customdata[2]:.2f}}</b><br>" +
                  f"🩺 {f2_ua}: <b>%{{customdata[3]:.2f}}</b><br>" +
                  "<extra></extra>"
)

fig_cluster_map.update_layout(
    title_text=f"🌐 Світовий розподіл рівнів життя за версією ШІ (GMM Clusters, {TARGET_YEAR})",
    title_x=0.5,
    title_font=dict(size=20, color="#ffffff"),
    template=PLOT_TEMPLATE,            
    geo=dict(
        showframe=False,
        showcoastlines=True, coastlinecolor="rgba(255, 255, 255, 0.3)",
        projection_type='natural earth',
        bgcolor='rgba(0,0,0,0)',
        lakecolor='#0a0a0a',
        landcolor='#1a1a1a'
    ),
    legend=dict(
        title="Категорії (Кластери)",
        yanchor="bottom", y=0.05, 
        xanchor="left", x=0.05,
        bgcolor="rgba(0,0,0,0.5)",
        bordercolor="#444", borderwidth=1
    ),
    width=1300, height=750,
    margin=dict(l=0, r=0, b=0, t=60)
)

print("Красивий Вивід - Кластерна мапа світу:")
fig_cluster_map.show()

print("\n📊 Технічний Вивід - Аналітика сформованих кластерів:")
cluster_stats = df.groupby('Cluster_Name').agg(
    count=(COUNTRY_COL, 'count'),
    mean_idx=(TARGET_METRIC, 'mean'),
    min_idx=(TARGET_METRIC, 'min'),
    max_idx=(TARGET_METRIC, 'max')
).reset_index().rename(columns={
    'count': 'Кількість країн',
    'mean_idx': 'Середній індекс',
    'min_idx': 'Мін. індекс',
    'max_idx': 'Макс. індекс'
}).sort_values('Середній індекс', ascending=False)

display(cluster_stats.style.background_gradient(cmap='YlGnBu', subset=['Середній індекс'])\
        .set_properties(**TABLE_PROPS).format(precision=3))

🗺️ Ініціалізація картографічного модуля (Choropleth Map)...
   ✅ Геодані та мітки кластерів знайдено. Будуємо проєкцію...

Красивий Вивід - Кластерна мапа світу:



📊 Технічний Вивід - Аналітика сформованих кластерів:


,Cluster_Name,Кількість країн,Середній індекс,Мін. індекс,Макс. індекс
0,Високий рівень,23,6.846,5.308,7.804
2,Середній рівень,73,5.799,3.694,7.530
1,Низький рівень,41,4.345,1.859,5.840


**13. Дослідити вплив:**

**13.9.\*\* Класифікація щастя:**

**14. Висновок:**